# 📱 SME Finance Copilot — Qwen 2.5 0.5B Mobile LoRA Training (Free T4 GPU)

This notebook fine-tunes **Qwen-2.5-0.5B-Instruct** on verified financial examples for mobile & low-resource CPU devices.

### Specifications:
- **Base Model**: `Qwen/Qwen2.5-0.5B-Instruct` (490M parameters)
- **Training Time**: ~3 minutes on free Google Colab T4 GPU
- **Output Size**: ~15 MB LoRA adapter (`qwen-0.5b-lora.zip`)
- **Runtime Target**: Pure CPU/mobile execution in **~350–450 MB RAM**

---

### Step 1: Install Dependencies

In [ ]:
!pip install -q torch transformers peft bitsandbytes datasets trl accelerate

### Step 2: Upload Dataset (`colab_dataset.zip`)
Run this cell and select `colab_dataset.zip` from your local `dataset/` directory (~2.2 MB).

In [ ]:
from google.colab import files
import os

print("Please select 'colab_dataset.zip' (from your project dataset/ folder):")
uploaded = files.upload()

for fn in uploaded.keys():
    if fn.endswith('.zip'):
        !unzip -o {fn}
        print(f"Extracted {fn} successfully!")

### Step 3: Run Mobile QLoRA Training (~3 Minutes)

In [ ]:
import os
import shutil
import gc
import torch

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

from datasets import load_dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer
from google.colab import files

model_id = "Qwen/Qwen2.5-0.5B-Instruct"
output_dir = "qwen-0.5b-lora"

# 1. 4-Bit NF4 Quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

# 2. Tokenizer and 4-Bit Base Model
print("[1/5] Loading Qwen 2.5 0.5B base model...")
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map={"": 0},
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)
model.gradient_checkpointing_enable()
model.config.use_cache = False

# 3. Configure LoRA for 0.5B architecture
print("[2/5] Configuring LoRA adapters...")
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

# 4. Load ChatML Dataset
print("[3/5] Loading ChatML dataset...")
train_dataset = load_dataset("json", data_files="finetune_train.jsonl", split="train")

def format_chatml(batch):
    formatted = []
    for conversation in batch["messages"]:
        text = tokenizer.apply_chat_template(conversation, tokenize=False, add_generation_prompt=False)
        formatted.append(text)
    return {"text": formatted}

train_dataset = train_dataset.map(format_chatml, batched=True, load_from_cache_file=False)

# 5. SFTTrainer Execution (~3 minutes on Colab T4)
print("[4/5] Starting fine-tuning loop...")
sft_config = SFTConfig(
    output_dir=output_dir,
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=3e-4,
    lr_scheduler_type="cosine",
    warmup_steps=20,
    logging_steps=10,
    save_strategy="epoch",
    save_total_limit=1,
    dataloader_pin_memory=True,
    fp16=False,
    bf16=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="paged_adamw_8bit",
    report_to="none",
    max_length=1024,
    dataset_text_field="text",
    eval_strategy="no",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    processing_class=tokenizer,
    args=sft_config,
)

trainer.train()

# 6. Save Adapter & Download
print("[5/5] Saving adapter and preparing download...")
trainer.model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

shutil.make_archive("qwen-0.5b-lora", "zip", output_dir)
print("Download started for qwen-0.5b-lora.zip (~15 MB)!")
files.download("qwen-0.5b-lora.zip")


### Step 4: After Download
1. Extract `qwen-0.5b-lora.zip` into `models/qwen-0.5b-mobile/` (or `models/qwen-0.5b-lora/`).
2. It is ready for instant, offline local inference in ~450 MB RAM on any CPU or phone!